Reading the csv file

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

#path.cwd = current working directory
#.parents[0] = parent of current working directory
PROJECT_ROOT = Path.cwd().resolve().parents[0]

#input folder 
RAW_DIR = PROJECT_ROOT / "data"/ "raw"

#output folder for processed dataset that will be saved
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

#create processed folder if does not exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

path = RAW_DIR / "Superstore.csv"

df = pd.read_csv(path, sep=";", engine="python")

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace("-", "_")
)
df.info()
df.head()
df.columns


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   row_id         9994 non-null   int64  
 1   order_id       9994 non-null   object 
 2   order_date     9994 non-null   object 
 3   customer_id    9994 non-null   object 
 4   customer_name  9994 non-null   object 
 5   segment        9994 non-null   object 
 6   city           9994 non-null   object 
 7   state          9994 non-null   object 
 8   product_id     9994 non-null   object 
 9   category       9994 non-null   object 
 10  sub_category   9994 non-null   object 
 11  product_name   9994 non-null   object 
 12  sales          9994 non-null   float64
 13  quantity       9994 non-null   int64  
 14  discount       9994 non-null   float64
 15  profit         9994 non-null   float64
dtypes: float64(3), int64(2), object(11)
memory usage: 1.2+ MB


Index(['row_id', 'order_id', 'order_date', 'customer_id', 'customer_name',
       'segment', 'city', 'state', 'product_id', 'category', 'sub_category',
       'product_name', 'sales', 'quantity', 'discount', 'profit'],
      dtype='object')

In [20]:
#convert string to real dates
date_cols = ["order_date"]

for col in date_cols:
    if col in df.columns:
        df[col]= pd.to_datetime(df[col], errors = "coerce")

#convert to real numbers
numeric_cols = ["sales", "quantity", "discount", "profit"]
df[numeric_cols] = df[numeric_cols].round().astype(int)

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors = "coerce")
        
             

In [21]:
#remove duplicates
df = df.drop_duplicates()
df = df.dropna(subset=["order_date", "sales"])
#remove impossible/invalid values (negatives or zero quantity)
df = df[(df["sales"] > 0) & (df["quantity"] >0)]

In [22]:
df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.to_period("M").astype(str)
df["order_dayofweek"] = df["order_date"].dt.day_name()

In [23]:
output_path = PROCESSED_DIR / "superstore_clean.csv"
df.to_csv(output_path, index = False)
output_path

PosixPath('/Users/traceytrinitasidharta/Desktop/ForeSight/data/processed/superstore_clean.csv')

In [24]:
print(df[["order_date", "sales"]].isna().sum())
print((df["sales"] <= 0).sum(), (df["quantity"] <= 0).sum())
print(df.dtypes)
print(df[["sales", "quantity", "discount", "profit"]].describe())

order_date    0
sales         0
dtype: int64
0 0
row_id                      int64
order_id                   object
order_date         datetime64[ns]
customer_id                object
customer_name              object
segment                    object
city                       object
state                      object
product_id                 object
category                   object
sub_category               object
product_name               object
sales                       int64
quantity                    int64
discount                    int64
profit                      int64
order_year                  int64
order_month                object
order_dayofweek            object
dtype: object
              sales     quantity     discount       profit
count   9994.000000  9994.000000  9994.000000  9994.000000
mean     230.416250     3.789574     0.085651    31.304583
std      623.406673     2.225110     0.279863   238.568628
min        1.000000     1.000000     0.000000 -6600.000

In [25]:
#ds: datetime column
#y: target sales
daily_sales = (
    df.groupby(["order_date", "category"], as_index=False)["sales"]
      .sum()
      .rename(columns={"order_date": "ds", "sales":"y"})
)
#save for forecasting notebook / forecasting tab
daily_sales_path = PROCESSED_DIR / "daily_sales.csv"
daily_sales.to_csv(daily_sales_path, index=False)
daily_sales_path

PosixPath('/Users/traceytrinitasidharta/Desktop/ForeSight/data/processed/daily_sales.csv')

In [27]:

daily_sales_subcat = (
    df.groupby(["order_date", "category", "sub_category"], as_index=False)["sales"]
    .sum()
    .rename(columns={"order_date": "ds", "sales":"y"})
)

daily_sales_subcat_path = PROCESSED_DIR / "daily_sales_subcategory.csv"
daily_sales_subcat.to_csv(daily_sales_subcat_path, index=False)
daily_sales_subcat_path

display(daily_sales_subcat.head())

,ds,category,sub_category,y
0,2014-01-03,Office Supplies,Paper,16
1,2014-01-04,Office Supplies,Binders,4
2,2014-01-04,Office Supplies,Labels,12
3,2014-01-04,Office Supplies,Storage,273
4,2014-01-05,Office Supplies,Art,20
